# EAUI 2026 - Análisis Completo con SHAP

Carga, procesamiento, modelado y análisis SHAP del modelo de predicción GSE.

In [ ]:
import pyreadstat
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías importadas')


## 1. Carga y Procesamiento de Datos

In [ ]:
# Cargar
df, meta = pyreadstat.read_sav("/Users/tomas/github/eaui_subtel/data/sav/2026.sav")
print(f'✓ {df.shape[0]:,} registros × {df.shape[1]} columnas')

# Calcular GSE
def educacion_grupo(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3: return 'basica'
    elif e <= 7: return 'media'
    elif e <= 9: return 'tecnica'
    else: return 'universitaria'

gse_map = {
    (1,'basica'):'E', (1,'media'):'E', (1,'tecnica'):'D', (1,'universitaria'):'D',
    (2,'basica'):'E', (2,'media'):'D', (2,'tecnica'):'D', (2,'universitaria'):'C3',
    (3,'basica'):'D', (3,'media'):'C3', (3,'tecnica'):'C3', (3,'universitaria'):'C2',
    (4,'basica'):'C3', (4,'media'):'C2', (4,'tecnica'):'C2', (4,'universitaria'):'C1',
    (5,'basica'):'C2', (5,'media'):'C1', (5,'tecnica'):'C1', (5,'universitaria'):'AB',
    (6,'basica'):'C1', (6,'media'):'AB', (6,'tecnica'):'AB', (6,'universitaria'):'AB',
}

eg = df['A10'].apply(educacion_grupo)
df['gse'] = pd.Categorical(
    df['A11'].combine(eg, lambda o, e: np.nan if pd.isna(o) or e is None else gse_map.get((int(o), e), np.nan)),
    categories=['AB','C1','C2','C3','D','E'], ordered=True
)

# Renombrar y recodificar
rename = {'Q1_1':'edad', 'Q1_2':'sexo', 'ZONA':'zona', 'P11':'pago_internet', 'Q7_4':'pago_movil', 'Q10':'frecuencia', 'Q11':'horas_diarias'}
df = df.rename(columns=rename)

df['sexo'] = df['sexo'].map({1:'Hombre', 2:'Mujer'})
df['zona'] = df['zona'].map({1:'Urbana', 2:'Rural'})
df['uso_smartphone'] = df['Q7'].map({1:'Sí', 2:'No'})
df['frecuencia'] = df['frecuencia'].map({1:'Diario', 2:'Frecuente', 3:'Ocasional', 4:'Muy ocasional'})
df['horas_diarias'] = df['horas_diarias'].map({1:0.5, 2:1.5, 3:3, 4:5})

print('✓ Datos procesados')


## 2. Features de Ingeniería

In [ ]:
# Número de dispositivos
dispositivos = [c for c in df.columns if c.startswith('P3_')][:7]
df['n_dispositivos'] = df[[c for c in dispositivos if c in df.columns]].fillna(0).sum(axis=1)

# Actividades online
actividades = [c for c in df.columns if c.startswith('Q21_')][:8]
df['n_actividades'] = df[[c for c in actividades if c in df.columns]].fillna(0).sum(axis=1)

# Intensidad de uso
df['intensidad'] = (df['frecuencia'].map({'Diario':4, 'Frecuente':3, 'Ocasional':2, 'Muy ocasional':1}).fillna(2) * 
                    df['horas_diarias'].fillna(1))

print('✓ Features de ingeniería:')
print(f'  n_dispositivos (media): {df["n_dispositivos"].mean():.2f}')
print(f'  n_actividades (media): {df["n_actividades"].mean():.2f}')
print(f'  intensidad (media): {df["intensidad"].mean():.2f}')


## 3. Preparación y Entrenamiento del Modelo

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Seleccionar features
features_num = ['edad', 'n_dispositivos', 'n_actividades', 'intensidad', 'pago_internet', 'pago_movil']
features_cat = ['sexo', 'zona', 'uso_smartphone']

# Dataset limpio
df_modelo = df[features_num + features_cat + ['gse']].dropna(subset=['gse']).copy()

# Imputar
for col in features_num:
    df_modelo[col] = df_modelo[col].fillna(df_modelo[col].median())
    
for col in features_cat:
    modo = df_modelo[col].mode()
    df_modelo[col] = df_modelo[col].fillna(modo[0] if len(modo) > 0 else 'Unknown')

# Codificar categóricas
le_dict = {}
for col in features_cat:
    le = LabelEncoder()
    df_modelo[col + '_encoded'] = le.fit_transform(df_modelo[col].astype(str))
    le_dict[col] = le

# Preparar datos
feature_cols = features_num + [c + '_encoded' for c in features_cat]
X = df_modelo[feature_cols].values
y = df_modelo['gse'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'✓ Dataset: {X.shape[0]:,} registros, {X.shape[1]} features')
print(f'✓ Train: {len(X_train):,}, Test: {len(X_test):,}')

# Entrenar modelo
print('\nEntrenando Random Forest...')
rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=15, 
    random_state=42, 
    n_jobs=-1, 
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)

y_pred = rf_model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)

print(f'✓ Accuracy: {accuracy:.4f}')
print(f'\nReporte de clasificación:')
print(classification_report(y_test, y_pred, target_names=rf_model.classes_))


## 4. Análisis de Feature Importance

In [ ]:
# Feature importance tradicional
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top 10 Features por Importancia:')
print(importance_df.head(10).to_string(index=False))

# Visualizar
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=rf_model.classes_,
            yticklabels=rf_model.classes_,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Matriz de Confusión', fontweight='bold', fontsize=11)
axes[0].set_ylabel('GSE Real')
axes[0].set_xlabel('GSE Predicho')

# Top features
top_features = importance_df.head(10)
axes[1].barh(range(len(top_features)), top_features['Importance'].values, 
             color='steelblue', edgecolor='black')
axes[1].set_yticks(range(len(top_features)))
axes[1].set_yticklabels(top_features['Feature'].values, fontsize=9)
axes[1].set_title('Top 10 Features - Feature Importance', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Importancia')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('✓ Visualizaciones generadas')


## 5. Análisis SHAP

In [ ]:
import shap

print('Calculando SHAP values...')

# Crear explicador
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_test_scaled)

# Para multiclase, usar clase mayoritaria
class_counts = [np.sum(y_test == c) for c in rf_model.classes_]
class_majority_idx = np.argmax(class_counts)
clase_mayoritaria = rf_model.classes_[class_majority_idx]

print(f'✓ SHAP calculados')
print(f'  Clase analizada: {clase_mayoritaria}')
print(f'  Base value: {explainer.expected_value[class_majority_idx]:.4f}')

# Importancia SHAP (usando valores de la clase mayoritaria)
shap_vals_clase = np.array(shap_values[class_majority_idx])
shap_importance = np.abs(shap_vals_clase).mean(axis=0)

shap_importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'SHAP_Importance': shap_importance
}).sort_values('SHAP_Importance', ascending=False)

print(f'\nTop 10 Features por SHAP:')
print(shap_importance_df.head(10).to_string(index=False))

# Guardar ranking
shap_importance_df.to_csv('shap_ranking.csv', index=False)
print('\n✓ Ranking SHAP guardado en shap_ranking.csv')


## 6. Comparación SHAP vs Feature Importance

In [ ]:
# Comparación lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top_n = 10

# Feature Importance
top_fi = importance_df.head(top_n)
axes[0].barh(range(len(top_fi)), top_fi['Importance'].values, 
             color='steelblue', edgecolor='black')
axes[0].set_yticks(range(len(top_fi)))
axes[0].set_yticklabels(top_fi['Feature'].values, fontsize=9)
axes[0].set_xlabel('Importancia')
axes[0].set_title('Feature Importance Tradicional', fontweight='bold', fontsize=11)
axes[0].invert_yaxis()

# SHAP
top_shap = shap_importance_df.head(top_n)
axes[1].barh(range(len(top_shap)), top_shap['SHAP_Importance'].values, 
             color='coral', edgecolor='black')
axes[1].set_yticks(range(len(top_shap)))
axes[1].set_yticklabels(top_shap['Feature'].values, fontsize=9)
axes[1].set_xlabel('SHAP Mean |Value|')
axes[1].set_title('Importancia SHAP', fontweight='bold', fontsize=11)
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('✓ Comparación completada')


## 7. Resumen del Análisis

In [ ]:
print('\n' + '='*70)
print('ANÁLISIS SHAP COMPLETADO EXITOSAMENTE')
print('='*70)

print(f'\n📊 DATASET:')
print(f'   Total inicial: {len(df):,} registros')
print(f'   Utilizado para modelo: {len(df_modelo):,} registros')
print(f'   Test: {len(X_test):,} muestras')

print(f'\n🎯 MODELO:')
print(f'   Tipo: Random Forest (100 árboles, max_depth=15)')
print(f'   Accuracy: {accuracy:.4f} ({int(accuracy*len(X_test))}/{len(X_test)} correctas)')
print(f'   Clases objetivo: {', '.join(rf_model.classes_)}')

print(f'\n📈 FEATURES:')
print(f'   Total: {len(feature_cols)}')
print(f'   Numéricos: {len(features_num)}')
print(f'   Categóricos: {len(features_cat)}')

print(f'\n🔝 TOP 5 FEATURES (SHAP):')
for i, row in shap_importance_df.head(5).iterrows():
    print(f'   {i+1}. {row["Feature"]:<25} {row["SHAP_Importance"]:.6f}')

print(f'\n✅ ARCHIVOS GENERADOS:')
print(f'   • eaui2026_v4.ipynb (este notebook)')
print(f'   • shap_ranking.csv (ranking completo)')
print(f'   • ANALISIS_SHAP.md (interpretación)')
print(f'   • informe_shap_analisis.html (reporte visual)')
print(f'   • ROBUSTEZ_Y_MANTENIMIENTO.md (garantías)')

print(f'\n{'='*70}')
print('✓ ANÁLISIS COMPLETADO CORRECTAMENTE')
print('='*70)
